# Personne A — Compagnies (J3 §1–2, J4 test n°1, onglet « Compagnies »)

**Question** : les compagnies se valent-elles face au retard ?

| Livrable | Où |
|---|---|
| J3 §1 : asymétrie de `arrival_delay` en Spark SQL | ce notebook |
| J3 §2 : table Gold `airline_ranking` (classement + Pareto) | `/lakehouse/gold/airline_ranking` |
| J4 test n°1 : Kruskal-Wallis du retard entre compagnies | ce notebook |
| Onglet « Compagnies » du dashboard | `app/sections/compagnies.py` |
| Assemblage final de `J3_...ipynb` et `J4_...ipynb` (fin de journée) | voir le doc de répartition |

**Règles communes** (détail dans `docs/repartition_J3_J4.md`)
- Ne modifiez que ce notebook et votre fichier `app/sections/compagnies.py` : aucun conflit git possible.
- Les tables Gold respectent **exactement** les noms et colonnes du contrat : le dashboard et l'assemblage final en dépendent. `save_gold(...)` le vérifie.
- Taux de retard = part des vols **arrivés** avec `arrival_delay >= 15` (vue `v_arrives`). Taux d'annulation = sur **tous** les vols (`v_silver`).
- Octobre a des codes aéroport non valides : pour toute analyse par aéroport, filtrez `airport_code_valid`.
- Chaque résultat est suivi d'une interprétation en Markdown (comme en J2). Committez le notebook **exécuté**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("A_compagnies").getOrCreate()
sns.set_theme(style="whitegrid", context="notebook")

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
# Vols arrivés : les seuls qui ont un arrival_delay (même périmètre que l'EDA de J2)
spark.sql("""
    SELECT * FROM v_silver
    WHERE NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
""").createOrReplaceTempView("v_arrives")
print("v_silver :", spark.table("v_silver").count(), "vols | v_arrives :", spark.table("v_arrives").count(), "vols arrivés")


def save_gold(df, name, contrat, n_files=1):
    """Vérifie les colonnes du contrat (docs/repartition_J3_J4.md) puis écrit /lakehouse/gold/<name> en Delta."""
    manquantes = [c for c in contrat if c not in df.columns]
    assert not manquantes, f"Colonnes du contrat manquantes pour {name} : {manquantes}"
    path = f"/lakehouse/gold/{name}"
    autres = [c for c in df.columns if c not in contrat]
    (df.select(*contrat, *autres).coalesce(n_files)
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path))
    n = spark.read.format("delta").load(path).count()
    print(f"Table Gold écrite : {path} ({n:,} lignes)")

## J3 §1 — Asymétrie : moyenne vs médiane et IQR

L'EDA de J2 (étape 0) l'a déjà fait en PySpark ; ici on le refait **en Spark SQL** sur `v_arrives` (exigence de J3) : `COUNT(*)`, `AVG`, `STDDEV`, `SKEWNESS`, `PERCENTILE_APPROX(arrival_delay, array(0.25, 0.5, 0.75, 0.9, 0.95))`, `MAX`.
Bonus utile pour le dashboard : le même résumé **par compagnie**, pour montrer que l'asymétrie existe dans chaque groupe.

In [ ]:
df_dist_summary = spark.sql("""
    -- TODO : résumé statistique de arrival_delay
    SELECT ...
    FROM v_arrives
""")
df_dist_summary.show(truncate=False)

**Interprétation** *(à compléter)* : que disent la skewness et l'écart moyenne / médiane ? Pourquoi le dashboard affiche-t-il un **taux de retard ≥ 15 min** et une **médiane + IQR** plutôt qu'une moyenne ?

## J3 §2 — Table Gold `airline_ranking` : classement et Pareto

Contrat (une ligne par compagnie) :

| colonne | définition |
|---|---|
| `airline`, `airline_name` | code IATA et nom |
| `n_vols` | vols arrivés |
| `n_retards` | vols arrivés avec `arrival_delay >= 15` |
| `taux_retard` | `100 * n_retards / n_vols` |
| `retard_median`, `retard_p25`, `retard_p75`, `retard_iqr`, `retard_p90` | quantiles de `arrival_delay` (minutes) |
| `rang` | `DENSE_RANK() OVER (ORDER BY taux_retard DESC)` (1 = la plus en retard) |
| `part_cumulee_retards` | Pareto, en % : `SUM(n_retards) OVER (ORDER BY n_retards DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(n_retards) OVER ()` |

Écrivez une requête avec un CTE (`WITH agg AS (...) SELECT ..., DENSE_RANK() ..., SUM(...) OVER ...`).

In [ ]:
AIRLINE_RANKING_COLS = ["airline", "airline_name", "n_vols", "n_retards", "taux_retard", "retard_median",
                        "retard_p25", "retard_p75", "retard_iqr", "retard_p90", "rang", "part_cumulee_retards"]

df_gold_1 = spark.sql("""
    WITH agg AS (
        -- TODO : agrégation par compagnie
        SELECT ...
        FROM v_arrives
        GROUP BY ...
    )
    SELECT *,
           -- TODO : rang et part cumulée (Pareto)
           ...
    FROM agg
    ORDER BY rang
""")
save_gold(df_gold_1, "airline_ranking", AIRLINE_RANKING_COLS)
df_gold_1.show(20, truncate=False)

**Interprétation** *(à compléter)* : écart entre la meilleure et la pire compagnie ; la loi 80/20 s'applique-t-elle (combien de compagnies concentrent 80 % des vols en retard) ? Est-ce lié à leur taille ?

## J4 test n°1 — Kruskal-Wallis : le retard diffère-t-il entre compagnies ?

- $H_0$ : la distribution de `arrival_delay` est la même pour toutes les compagnies. $H_1$ : au moins une compagnie diffère.
- Pourquoi Kruskal-Wallis : k ≥ 3 groupes, variable très asymétrique (§1), donc pas d'ANOVA.
- Ne rapatriez que `airline` et `arrival_delay` (`select` côté Spark avant `.toPandas()`).
- Avec ~280 000 vols, la p-value sera minuscule : ajoutez une **taille d'effet** (ε² = H / (n − 1)) pour dire si l'écart compte en pratique.

In [ ]:
# TODO : extraire airline + arrival_delay, un groupe par compagnie, stats.kruskal(*groupes)
df_test = ...
H, p_value = ...
print(f"Kruskal-Wallis : H = {H:.1f}, p = {p_value:.2e}, epsilon² = {H / (len(df_test) - 1):.4f}")

**Conclusion du test** *(à compléter)* : on rejette / ne rejette pas $H_0$ au seuil de 5 %, taille d'effet, ce que ça signifie pour un voyageur.